# Part A Question 1: Tokenization and Filtering

This notebook demonstrates word tokenization using the split function, regular expressions, and NLTK `word_tokenize`. It also removes NLTK stop words and punctuation from `Data_1.txt`.

In [ ]:
import re
import string
from dataclasses import dataclass
from pathlib import Path
from pprint import pprint

import nltk
from nltk.data import find
from nltk.tokenize import word_tokenize

In [ ]:
DATA_PATH = Path('data/Data_1.txt')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/Data_1.txt')


def load_text(path: Path) -> str:
    """Read a UTF-8 text file and normalize line breaks to spaces."""

    return path.read_text(encoding='utf-8').replace('\r\n', '\n').replace('\n', ' ')


text = load_text(DATA_PATH)
text

## 1. Split Function Tokenization

In [ ]:
def tokenize_with_split(text: str) -> list[str]:
    """Tokenize by splitting only on whitespace."""

    return text.split()


split_tokens = tokenize_with_split(text)
pprint(split_tokens)

## 2. Regular Expression Tokenization

In [ ]:
def tokenize_with_regex(text: str) -> list[str]:
    """Tokenize into words and punctuation with a regular expression."""

    return re.findall(r'\b\w+\b|[^\w\s]', text)


regex_tokens = tokenize_with_regex(text)
pprint(regex_tokens)

## 3. NLTK Tokenization

In [ ]:
def tokenize_with_nltk(text: str) -> list[str]:
    """Tokenize with NLTK's word tokenizer using Punkt sentence rules."""

    try:
        find('tokenizers/punkt_tab')
    except LookupError:
        nltk.download('punkt_tab', quiet=True)

    return word_tokenize(text)


nltk_tokens = tokenize_with_nltk(text)
pprint(nltk_tokens)

## 4. Stop Words and Punctuation Removal

In [ ]:
@dataclass(frozen=True)
class FilteringResult:
    """Tokens kept after filtering and stop words found in the source text."""

    filtered_tokens: list[str]
    stopwords_found: list[str]


def get_stopwords() -> set[str]:
    """Return NLTK English stop words, downloading the corpus if needed."""

    try:
        find('corpora/stopwords')
    except LookupError:
        nltk.download('stopwords', quiet=True)

    from nltk.corpus import stopwords

    return set(stopwords.words('english'))


def remove_stopwords_and_punctuation(tokens: list[str]) -> FilteringResult:
    """Remove English stop words and punctuation from token boundaries."""

    stopwords = get_stopwords()
    punctuation = set(string.punctuation)
    filtered_tokens: list[str] = []
    stopwords_found: set[str] = set()

    for token in tokens:
        cleaned_token = token.strip(string.punctuation)
        normalized = cleaned_token.casefold()

        if normalized in stopwords:
            stopwords_found.add(normalized)
            continue

        if not cleaned_token or all(character in punctuation for character in cleaned_token):
            continue

        filtered_tokens.append(cleaned_token)

    return FilteringResult(filtered_tokens=filtered_tokens, stopwords_found=sorted(stopwords_found))


filtering_result = remove_stopwords_and_punctuation(nltk_tokens)
print('Stop words found:')
pprint(filtering_result.stopwords_found)
print('\nFiltered tokens:')
pprint(filtering_result.filtered_tokens)